# FDS Challenge: Pokémon Battle Prediction — final submissions

Classical machine learning only (no neural networks). Given the first 30 turns of a Gen 2 OU battle,
we predict whether player 1 wins. This notebook clones the
[GitHub repository](https://github.com/MarcoValerioQuinti/pokemon-fds) at a fixed commit and
regenerates our **3 selected final submissions**.

**Pipeline** (code in `src/`; methodology in the repository README):

1. **Battle-state features** (`features.py`): about 280 features from a Gen 2 damage engine (HP, KOs,
   status, trajectory, damage matrix between the alive Pokémon, Spikes, sleep, win conditions), plus
   bag-of-words of Pokémon and moves.
2. **Monte Carlo features** (`simulate.py`): every battle is played to the end 100 times from turn 30.
3. **Recursive features** (`recursive.py`): roll-outs of a gradient-boosted turn-to-turn transition model.
4. **Snapshot augmentation** (`stacking.py`): training battles truncated at turns 24/26/28 are added
   inside the training folds, with weight 0.5.
5. **Stacking** (`final_model.py`): out-of-fold predictions of the base models (stratified 5-fold,
   seed 42), combined by a meta-learner.

**No test information is learned.** Pokédex, Hidden Power types, co-occurrence, move statistics,
bag-of-words vocabulary, the Monte Carlo context and the transition model are fitted on the training
battles only; the test battles are only transformed.

**Settings:**
- Accelerator **None**: about 1 hour on Kaggle's 4 CPU cores. The run is pinned to the CPU (`DEVICE`
  below), so the selected accelerator never changes the predictions. A GPU would only speed up
  XGBoost and CatBoost, with slightly different results; a TPU is not used by any of these libraries.
- Internet **on**, needed to clone the repository.

### Setup: clone the repository at the fixed commit

In [ ]:
import glob
import os
import subprocess
import sys
import time

import pandas as pd

REPO = "https://github.com/MarcoValerioQuinti/pokemon-fds.git"
COMMIT = "1cbf9f3"  # exact version of the code that produced the selected submissions
DEVICE = "cpu"  # "cpu" reproduces the selected submissions; "auto" uses an NVIDIA GPU for XGBoost/CatBoost


def find_data_dir():
    """Competition data: mounted under /kaggle/input on Kaggle, POKEMON_DATA or ./data elsewhere."""
    hits = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
    if hits:
        return os.path.dirname(hits[0])
    return os.environ.get("POKEMON_DATA", os.path.abspath("data"))


DATA_DIR = find_data_dir()
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
CODE_DIR = os.path.join(OUT_DIR, "pokemon-fds")
WORK_DIR = os.path.join(OUT_DIR, "outputs")  # feature caches, simulations, out-of-fold predictions
if not os.path.isdir(CODE_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO, CODE_DIR], check=True)
subprocess.run(["git", "-C", CODE_DIR, "checkout", "--quiet", COMMIT], check=True)
os.environ.update(POKEMON_DATA=DATA_DIR, POKEMON_OUT=WORK_DIR, POKEMON_DEVICE=DEVICE)

import catboost, lightgbm, numpy, sklearn, xgboost
print("code:", subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
print(f"data: {DATA_DIR} -> {sorted(os.listdir(DATA_DIR))}")
print(f"device: {DEVICE}; cores: {os.cpu_count()}; numpy {numpy.__version__}, scikit-learn {sklearn.__version__}, "
      f"lightgbm {lightgbm.__version__}, xgboost {xgboost.__version__}, catboost {catboost.__version__}")

In [ ]:
def run(script, *args):
    """Runs src/<script> of the cloned repository, streams its log and stops the notebook if it fails."""
    t0 = time.time()
    proc = subprocess.Popen([sys.executable, "-u", os.path.join(CODE_DIR, "src", script), *args],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if "Warning" not in line and "warnings.warn" not in line:  # hide library warnings
            print(line, end="")
    if proc.wait():
        raise RuntimeError(f"{script} failed with exit code {proc.returncode}")
    print(f"--- {script}: {(time.time() - t0) / 60:.1f} min")


def write_submission(config, n):
    """Aligns the predictions of `config` to sample_submission.csv and writes submission_<n>_<config>.csv."""
    pred = pd.read_csv(os.path.join(WORK_DIR, f"submission_{config}.csv"))
    sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
    sub = sample[["battle_id"]].merge(pred, on="battle_id", how="left")
    assert len(sub) == len(sample) and sub["player_won"].notna().all()
    sub["player_won"] = sub["player_won"].astype(int)
    path = os.path.join(OUT_DIR, f"submission_{n}_{config}.csv")
    sub.to_csv(path, index=False)
    print(f"{os.path.basename(path)}: {len(sub)} rows, player 1 predicted to win in {sub.player_won.mean():.1%}")
    return sub

### Pipeline

Every step caches its results in `outputs/`. The log shows the out-of-fold accuracy of each base model and the cross-validated accuracy of each configuration.

**Monte Carlo features** (15,000 battles + 30,000 training snapshots, 100 simulations each).

In [ ]:
run("simulate.py", "--n-sims", "100")

**Recursive roll-out features** (transition model trained on the training battles only).

In [ ]:
run("recursive.py")

**Battle-state features, out-of-fold predictions of all base models, the 3 configurations.**

In [ ]:
run("final_model.py", "--configs", "stack10,stack22,stack10_mean")

### Submission 1: `stack10`, the final 10-model stack

The main solution. Ten base models were chosen by a cost/benefit analysis of a 22-model zoo, keeping
the ones that make *different* errors:
- LightGBM, XGBoost and CatBoost trained with snapshot augmentation (on the full feature set and on the
  dense set without bag-of-words);
- ExtraTrees and Random Forest;
- LDA and two logistic regressions.

A logistic regression on the logits of their out-of-fold predictions combines them.

In [ ]:
write_submission("stack10", 1).head()

### Submission 2: `stack22`, the full 22-model stack

The same features and meta-learner, applied to the whole model zoo:
- the 10 models of submission 1;
- boosting without augmentation and Gaussian-noise variants;
- SVM, kNN, QDA and Gaussian-mixture classifiers.

It is a more redundant but larger ensemble: a hedge in case the pruning of submission 1 was
tuned too closely to the training folds.

In [ ]:
write_submission("stack22", 2).head()

### Submission 3: `stack10_mean`, the 10 models averaged

The 10 base models of submission 1, combined by a plain average of their logits instead of learned
weights. With no meta-learner to fit, it is the most robust to overfitting of the stacking layer.

In [ ]:
write_submission("stack10_mean", 3).head()

### Submitting

On the Kaggle website, open this notebook's **Output** and use **Submit to Competition** on each of
the three `submission_<n>_<config>.csv` files.

From the Kaggle CLI, with the version number of this notebook run:

```bash
kaggle competitions submit fds-pokemon-battles-prediction-2026 -k <user>/pokemon-fds-final -v <version> -f submission_1_stack10.csv -m "stack10"
```